In [1]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd


TASK_DIR = Path.cwd().parent
INPUT_DIR = TASK_DIR / "input"
OUTPUT_DIR = TASK_DIR / "output"

LONG_PATH = INPUT_DIR / "productivity_long.csv"
TRANSITIONS_PATH = INPUT_DIR / "productivity_transitions.csv"

INITIAL_PARAMS_PATH = OUTPUT_DIR / "initial_exponential_params.csv"
STAGE_PARAMS_PATH = OUTPUT_DIR / "stage_grw_params.csv"
COUNTS_PATH = OUTPUT_DIR / "fit_counts_by_destination_year.csv"
MANIFEST_PATH = OUTPUT_DIR / "fit_manifest.json"

MODEL_NAME = "Stagewise AR(1)-GRW"
MODEL_TAG = "stagewise_ar1_grw"

EPS = 0.49
Y = 20

STAGES = [
    {"stage": "years_1_4","start": 1,"end": 4},
    {"stage": "years_5_7","start": 5,"end": 7},
    {"stage": "years_8_20","start": 8, "end": 20,}]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:

productivity_long = pd.read_csv(LONG_PATH,dtype={"dblp_id": "string", "dblp": "string"})

transitions = pd.read_csv(TRANSITIONS_PATH,dtype={"dblp_id": "string", "dblp": "string"})

In [3]:
transitions["z_current"] = np.log(transitions["pubs_adj"] + EPS)

transitions["z_next"] = np.log(transitions["pubs_adj_next"] + EPS)


def destination_stage(age):
    for spec in STAGES:
        if spec["start"] <= age <= spec["end"]:
            return spec["stage"]

    return pd.NA


transitions["stage"] = (transitions["CareerAge_next"].astype("int64").map(destination_stage).astype("string"))

fit_counts_by_year = (transitions.groupby(["stage", "CareerAge_next"]).size().rename("n").reset_index().sort_values("CareerAge_next"))

In [4]:
q0 = (productivity_long.loc[productivity_long["CareerAge"].eq(0),"pubs_adj"].dropna().to_numpy(dtype=float))

alpha0 = float(q0.mean())
lambda0 = float(1 / alpha0) if alpha0 > 0 else np.inf

initial_params = pd.DataFrame([{"stage": "year_0","distribution": "exponential","loc": 0.0,"scale_alpha": alpha0,"rate_lambda": lambda0,"n": int(len(q0))}])

In [5]:
def fit_stage_grw(subset):
    x = subset["z_current"].to_numpy(dtype=float)
    y = subset["z_next"].to_numpy(dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    x_centered = x - x.mean()
    y_centered = y - y.mean()

    denominator = np.dot(x_centered, x_centered)

    beta = np.dot(x_centered, y_centered) / denominator

    log_gamma = y - beta * x

    mu_log_gamma = log_gamma.mean()
    sigma_log_gamma = np.sqrt(np.mean((log_gamma - mu_log_gamma) ** 2))

    fitted_log_mean = beta * x + mu_log_gamma
    centered_residual = y - fitted_log_mean

    beta_se = np.sqrt(sigma_log_gamma**2 / denominator)

    return {
        "n": int(len(x)),
        "ar_intercept": 0.0,
        "beta": float(beta),
        "beta_se": float(beta_se),
        "lognormal_mu": float(mu_log_gamma),
        "lognormal_sigma": float(sigma_log_gamma),
        "lognormal_scale": float(np.exp(mu_log_gamma)),
        "lognormal_median": float(np.exp(mu_log_gamma)),
        "lognormal_mean": float(np.exp(mu_log_gamma + 0.5 * sigma_log_gamma**2)),
        "centered_residual_mean": float(centered_residual.mean()),
        "centered_residual_sd": float(centered_residual.std(ddof=0))}

stage_rows = []

for spec in STAGES:
    subset = transitions.loc[transitions["stage"].eq(spec["stage"])].copy()
    stage_rows.append({**spec,**fit_stage_grw(subset)})

stage_params = pd.DataFrame(stage_rows)

display(stage_params)

,stage,start,end,n,ar_intercept,beta,beta_se,lognormal_mu,lognormal_sigma,lognormal_scale,lognormal_median,lognormal_mean,centered_residual_mean,centered_residual_sd
0,years_1_4,1,4,8511,0.0,0.451436,0.009723,0.857612,0.983639,2.357525,2.357525,3.824338,5.343054e-17,0.983639
1,years_5_7,5,7,6486,0.0,0.536658,0.010430,0.742392,0.911151,2.100955,2.100955,3.181937,-9.859520e-17,0.911151
2,years_8_20,8,20,19112,0.0,0.600853,0.005690,0.531023,0.903748,1.700672,1.700672,2.558455,-2.937049e-17,0.903748


In [6]:
expected_destination_years = set(range(1, Y + 1))

for spec in STAGES:
    stage = spec["stage"]

    expected = set(range(spec["start"],spec["end"] + 1))

    observed = set(transitions.loc[transitions["stage"].eq(stage),"CareerAge_next"].astype(int))

    missing = sorted(expected - observed)
    assert not missing

assert set(transitions["CareerAge_next"].astype(int)).issubset(expected_destination_years)

assert stage_params["ar_intercept"].eq(0).all()
assert np.isfinite(stage_params["beta"]).all()
assert np.isfinite(stage_params["lognormal_mu"]).all()
assert stage_params["lognormal_sigma"].ge(0).all()


display(stage_params[["stage","n","ar_intercept","beta","lognormal_mu","lognormal_sigma"]])

,stage,n,ar_intercept,beta,lognormal_mu,lognormal_sigma
0,years_1_4,8511,0.0,0.451436,0.857612,0.983639
1,years_5_7,6486,0.0,0.536658,0.742392,0.911151
2,years_8_20,19112,0.0,0.600853,0.531023,0.903748


In [7]:
initial_params.to_csv(INITIAL_PARAMS_PATH, index=False)
stage_params.to_csv(STAGE_PARAMS_PATH, index=False)
fit_counts_by_year.to_csv(COUNTS_PATH, index=False)